In [1]:
# Import required libraries (CUDA version)
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Import machine learning libraries
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from torch.utils.data import TensorDataset, DataLoader


In [2]:
# Load the dataset into a pandas DataFrame
df = pd.read_csv('../Sino_CT.csv')

# Convert the 'embedding' column from string to a list
df['embedding'] = df['embedding'].apply(eval)

# Map the 'label' column to 0 for abnormal ('0,1') and 1 for normal ('1,0')
df['label'] = df['label'].apply(lambda x: 1 if x == '1,0' else 0)

# Define the target column
target_column = 'label'

# Load the embeddings and target values
X = np.array(df['embedding'].tolist(), dtype=np.float32)
y = df[target_column].values.astype(np.float32)

# Reshape y to be (n_samples, 1) for consistency with the model output
y = y.reshape(-1, 1)

# Split the data with 80 percent for training and 20 percent for testing
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test_scaled, dtype=torch.float32)
y_train_tensor = torch.tensor(y_train, dtype=torch.float32)
y_test_tensor = torch.tensor(y_test, dtype=torch.float32)

# Create TensorDatasets and DataLoaders
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
test_dataset = TensorDataset(X_test_tensor, y_test_tensor)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

print("Data loaded and preprocessed successfully.")
print("X_train_tensor shape:", X_train_tensor.shape)
print("y_train_tensor shape:", y_train_tensor.shape)

Data loaded and preprocessed successfully.
X_train_tensor shape: torch.Size([5464, 1408])
y_train_tensor shape: torch.Size([5464, 1])


In [4]:
import torch
import torch.nn as nn

class TensorFlowToPyTorchModel(nn.Module):
    def __init__(self, input_features=1408):
        super(TensorFlowToPyTorchModel, self).__init__()
        
        # Define the sequential layers
        self.model = nn.Sequential(
            # First block: 1408 → 1536
            nn.Linear(input_features, 1536),
            nn.BatchNorm1d(1536),
            nn.ReLU(),
            nn.Dropout(p=0.4372695307748339),
            
            # Second block: 1536 → 512
            nn.Linear(1536, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(p=0.3417774049274107),
            
            # Third block: 512 → 896
            nn.Linear(512, 896),
            nn.BatchNorm1d(896),
            nn.ReLU(),
            nn.Dropout(p=0.4085042415826402),
            
            # Fourth block: 896 → 128
            nn.Linear(896, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(p=0.24257773824341827),
            
            # Output layer: 128 → 1 (no activation, BN, or dropout)
            nn.Linear(128, 1)
        )
    
    def forward(self, x):
        return self.model(x)

# Alternative block-based implementation for better organization
class TensorFlowToPyTorchModelBlocks(nn.Module):
    def __init__(self, input_features=1408):
        super(TensorFlowToPyTorchModelBlocks, self).__init__()
        
        # Define each block separately
        self.block1 = self._make_block(input_features, 1536, 0.4372695307748339)
        self.block2 = self._make_block(1536, 512, 0.3417774049274107)
        self.block3 = self._make_block(512, 896, 0.4085042415826402)
        self.block4 = self._make_block(896, 128, 0.24257773824341827)
        
        # Output layer
        self.output = nn.Linear(128, 1)
    
    def _make_block(self, in_features, out_features, dropout_rate):
        """Helper method to create a standard block"""
        return nn.Sequential(
            nn.Linear(in_features, out_features),
            nn.BatchNorm1d(out_features),
            nn.ReLU(),
            nn.Dropout(p=dropout_rate)
        )
    
    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        x = self.block4(x)
        x = self.output(x)
        return x

# Instantiate the model
model = TensorFlowToPyTorchModel()
print(model)


TensorFlowToPyTorchModel(
  (model): Sequential(
    (0): Linear(in_features=1408, out_features=1536, bias=True)
    (1): BatchNorm1d(1536, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU()
    (3): Dropout(p=0.4372695307748339, inplace=False)
    (4): Linear(in_features=1536, out_features=512, bias=True)
    (5): BatchNorm1d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU()
    (7): Dropout(p=0.3417774049274107, inplace=False)
    (8): Linear(in_features=512, out_features=896, bias=True)
    (9): BatchNorm1d(896, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU()
    (11): Dropout(p=0.4085042415826402, inplace=False)
    (12): Linear(in_features=896, out_features=128, bias=True)
    (13): BatchNorm1d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (14): ReLU()
    (15): Dropout(p=0.24257773824341827, inplace=False)
    (16): Linear(in_features=128, out_features=1, bi